<a href="https://colab.research.google.com/github/alexpanayi474/DSC511-MovieLens-Project/blob/main/TMDB_Enrichment_for_MovieLens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# TMDB Enrichment for MovieLens

This notebook enriches MovieLens movies with selected TMDB metadata. It saves checkpoint CSV files during the run so progress is not lost if Colab disconnects.

In [1]:
# Mount Google Drive so the MovieLens files and checkpoint files are available.
from google.colab import drive
drive.mount('/content/gdrive')

# Project data folder in Google Drive.
google_drive_path = "/content/gdrive/MyDrive/Colab Notebooks/Data/DSC_511_PROJECT/"

Mounted at /content/gdrive


In [2]:
# Core libraries for data handling, API requests, parallel execution, and checkpoints.
import os
import csv
import time
import threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests

In [3]:
# TMDB API key.
# Keep the key in one place so it is easy to replace or hide before submission.
TMDB_API_KEY = "55d206bac9cdb44a8069f01785a51884"

# Performance settings.
# SAMPLE_START controls which 10,000-movie batch runs.
# 0 = first 10,000 movies, 10000 = second 10,000, 20000 = third 10,000, etc.
# Change SAMPLE_START for the next batch after the current batch finishes.
SAMPLE_START = 10000

# SAMPLE_SIZE=10000 keeps each run manageable in Colab.
# Use SAMPLE_SIZE=None only when the full MovieLens enrichment is needed.
SAMPLE_SIZE = 10000
MAX_WORKERS = 8
SAVE_EVERY = 250

# Output files saved in Google Drive.
OUTPUT_DIR = Path(google_drive_path) / "enrichment_cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_FILE = OUTPUT_DIR / "tmdb_metadata_selected.csv"
ENRICHED_MOVIES_FILE = OUTPUT_DIR / "movies_enriched_selected.csv"

print("Checkpoint file:", CHECKPOINT_FILE)
print("Enriched output:", ENRICHED_MOVIES_FILE)

Checkpoint file: /content/gdrive/MyDrive/Colab Notebooks/Data/DSC_511_PROJECT/enrichment_cache/tmdb_metadata_selected.csv
Enriched output: /content/gdrive/MyDrive/Colab Notebooks/Data/DSC_511_PROJECT/enrichment_cache/movies_enriched_selected.csv


In [4]:
# Load MovieLens files.
# links.csv connects MovieLens movieId to TMDB and IMDb identifiers.
ratings_df = pd.read_csv(Path(google_drive_path) / "ratings.csv", usecols=["movieId"])
movies_df = pd.read_csv(Path(google_drive_path) / "movies.csv")
links_df = pd.read_csv(Path(google_drive_path) / "links.csv")

print("ratings rows:", len(ratings_df))
print("movies rows:", len(movies_df))
print("links rows:", len(links_df))

ratings rows: 33832162
movies rows: 86537
links rows: 86537


In [5]:
# Keep only movies that appear in the ratings data.
# This avoids API calls for movies that are not needed for the analysis.
rated_movie_ids = set(ratings_df["movieId"].dropna().astype(int).unique())

links_filtered = links_df[
    links_df["movieId"].isin(rated_movie_ids)
    & links_df["tmdbId"].notna()
].copy()

links_filtered["movieId"] = links_filtered["movieId"].astype(int)
links_filtered["tmdbId"] = links_filtered["tmdbId"].astype(int)

# A subset keeps the scrape practical for Colab runtime limits.
# SAMPLE_START selects the batch, and SAMPLE_SIZE controls the batch size.
if SAMPLE_SIZE is not None:
    links_filtered = links_filtered.iloc[SAMPLE_START:SAMPLE_START + SAMPLE_SIZE]

print("Unique rated movies:", len(rated_movie_ids))
print("Movies selected for TMDB enrichment:", len(links_filtered))
links_filtered.head()

Unique rated movies: 83239
Movies selected for TMDB enrichment: 10000


,movieId,imdbId,tmdbId
10026,33896,420251,33908
10027,33898,381270,14653
10028,33903,408777,276
10029,33905,49778,28000
10030,33912,78444,38731


In [6]:
# Selected output fields.
# Score and rating fields are intentionally excluded from the enrichment file.
FIELDNAMES = [
    "movieId",
    "tmdb_id",
    "imdb_id",
    "budget",
    "revenue",
    "runtime",
    "release_date",
    "release_year",
    "original_language",
    "genres_tmdb",
    "production_countries",
    "production_companies",
    "director",
    "top_cast",
    "writer",
    "overview",
    "keywords",
]

In [7]:
# Load previous checkpoint results if the file already exists.
# Rerunning the notebook continues from the missing movies only.
def load_checkpoint(path):
    if not path.exists():
        return pd.DataFrame(columns=FIELDNAMES), set()

    saved_df = pd.read_csv(path)
    completed_ids = set(saved_df["tmdb_id"].dropna().astype(int))
    return saved_df, completed_ids


# Append rows to the checkpoint file without rewriting the whole file every time.
def append_checkpoint(path, rows):
    if not rows:
        return

    write_header = not path.exists() or path.stat().st_size == 0
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES, extrasaction="ignore")
        if write_header:
            writer.writeheader()
        writer.writerows(rows)


saved_df, completed_tmdb_ids = load_checkpoint(CHECKPOINT_FILE)
print("Previously saved movies:", len(completed_tmdb_ids))

Previously saved movies: 9981


In [8]:
# One HTTP session per worker thread improves speed by reusing connections.
thread_local = threading.local()

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}


def get_session():
    if not hasattr(thread_local, "session"):
        session = requests.Session()
        session.headers.update(HEADERS)
        thread_local.session = session
    return thread_local.session

In [9]:
# Fetch selected TMDB metadata for one movie.
# append_to_response brings credits and keywords in the same request.
def fetch_tmdb_metadata(row, retries=3):
    movie_id = int(row["movieId"])
    tmdb_id = int(row["tmdbId"])
    imdb_id = None if pd.isna(row.get("imdbId")) else int(row["imdbId"])

    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"
    params = {
        "api_key": TMDB_API_KEY,
        "append_to_response": "credits,keywords",
    }

    for attempt in range(retries):
        try:
            response = get_session().get(url, params=params, timeout=20)

            # Missing TMDB pages are skipped.
            if response.status_code == 404:
                return None

            # Rate limits are retried with a short wait.
            if response.status_code == 429:
                time.sleep(2 + attempt * 3)
                continue

            response.raise_for_status()
            data = response.json()

            crew = data.get("credits", {}).get("crew", [])
            cast = data.get("credits", {}).get("cast", [])
            genres = data.get("genres", [])
            countries = data.get("production_countries", [])
            companies = data.get("production_companies", [])
            keywords = data.get("keywords", {}).get("keywords", [])

            directors = [p.get("name") for p in crew if p.get("job") == "Director"]
            writers = [p.get("name") for p in crew if p.get("job") in {"Writer", "Screenplay"}]

            release_date = data.get("release_date") or None
            release_year = int(release_date[:4]) if release_date and len(release_date) >= 4 else None

            return {
                "movieId": movie_id,
                "tmdb_id": tmdb_id,
                "imdb_id": imdb_id,
                "budget": data.get("budget") or None,
                "revenue": data.get("revenue") or None,
                "runtime": data.get("runtime") or None,
                "release_date": release_date,
                "release_year": release_year,
                "original_language": data.get("original_language"),
                "genres_tmdb": "|".join(g.get("name", "") for g in genres),
                "production_countries": "|".join(c.get("name", "") for c in countries),
                "production_companies": "|".join(c.get("name", "") for c in companies[:5]),
                "director": "|".join(directors[:3]) if directors else None,
                "top_cast": "|".join(p.get("name", "") for p in cast[:5]) if cast else None,
                "writer": "|".join(dict.fromkeys(writers[:5])) if writers else None,
                "overview": data.get("overview"),
                "keywords": "|".join(k.get("name", "") for k in keywords[:10]) if keywords else None,
            }

        except requests.RequestException:
            time.sleep(1 + attempt * 2)

    return None

In [10]:
# Build the pending list by removing movies already saved in the checkpoint.
pending_df = links_filtered[~links_filtered["tmdbId"].isin(completed_tmdb_ids)].copy()

print("Movies in current run:", len(links_filtered))
print("Already completed:", len(completed_tmdb_ids))
print("Pending movies:", len(pending_df))

Movies in current run: 10000
Already completed: 9981
Pending movies: 9999


In [11]:
# Run TMDB requests in parallel.
# This loop continues through the whole selected subset and saves progress every SAVE_EVERY successful rows.
rows = pending_df.to_dict("records")
buffer = []
success_count = 0
failed_count = 0
start_time = time.time()

print(f"Starting TMDB enrichment for {len(rows):,} pending movies...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(fetch_tmdb_metadata, row) for row in rows]

    for completed, future in enumerate(as_completed(futures), start=1):
        result = future.result()

        if result is None:
            failed_count += 1
        else:
            buffer.append(result)
            success_count += 1

        # Checkpoint saves partial progress before the full scrape completes.
        if len(buffer) >= SAVE_EVERY:
            append_checkpoint(CHECKPOINT_FILE, buffer)
            buffer = []
            elapsed_min = (time.time() - start_time) / 60
            print(f"Saved checkpoint | completed={completed:,} success={success_count:,} failed={failed_count:,} elapsed={elapsed_min:.1f} min")

        # Progress display for long runs.
        if completed % 500 == 0 or completed == len(rows):
            elapsed_min = (time.time() - start_time) / 60
            print(f"Progress {completed:,}/{len(rows):,} | success={success_count:,} failed={failed_count:,} elapsed={elapsed_min:.1f} min")

# Final checkpoint save for the remaining rows.
append_checkpoint(CHECKPOINT_FILE, buffer)

print("TMDB enrichment finished.")
print("Successful new rows:", success_count)
print("Failed or skipped rows:", failed_count)

Starting TMDB enrichment for 9,999 pending movies...
Saved checkpoint | completed=250 success=250 failed=0 elapsed=0.1 min
Progress 500/9,999 | success=499 failed=1 elapsed=0.2 min
Saved checkpoint | completed=501 success=500 failed=1 elapsed=0.2 min
Saved checkpoint | completed=754 success=750 failed=4 elapsed=0.3 min
Progress 1,000/9,999 | success=996 failed=4 elapsed=0.5 min
Saved checkpoint | completed=1,004 success=1,000 failed=4 elapsed=0.5 min
Saved checkpoint | completed=1,256 success=1,250 failed=6 elapsed=0.6 min
Progress 1,500/9,999 | success=1,493 failed=7 elapsed=0.7 min
Saved checkpoint | completed=1,507 success=1,500 failed=7 elapsed=0.7 min
Saved checkpoint | completed=1,760 success=1,750 failed=10 elapsed=0.8 min
Progress 2,000/9,999 | success=1,989 failed=11 elapsed=0.9 min
Saved checkpoint | completed=2,011 success=2,000 failed=11 elapsed=0.9 min
Saved checkpoint | completed=2,263 success=2,250 failed=13 elapsed=1.0 min
Progress 2,500/9,999 | success=2,487 failed=13 

In [12]:
# Load the full checkpoint after scraping.
# Duplicate tmdb_id rows are removed in case a run was restarted near a checkpoint.
tmdb_metadata_df = pd.read_csv(CHECKPOINT_FILE)
tmdb_metadata_df = tmdb_metadata_df.drop_duplicates(subset=["tmdb_id"], keep="last")

# Re-save the cleaned checkpoint.
tmdb_metadata_df.to_csv(CHECKPOINT_FILE, index=False)

print("Total saved TMDB rows:", len(tmdb_metadata_df))
tmdb_metadata_df.head()

Total saved TMDB rows: 19882


,movieId,tmdb_id,imdb_id,budget,revenue,runtime,release_date,release_year,original_language,genres_tmdb,production_countries,production_companies,director,top_cast,writer,overview,keywords
0,3,15602,113228,25000000.0,71518503.0,101.0,1995-12-22,1995,en,Romance|Comedy,United States of America,Lancaster Gate|Warner Bros. Pictures,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret|Sophia ...,Mark Steven Johnson,A family wedding reignites the ancient feud be...,fishing|sequel|old man|best friend|wedding|ita...
1,5,11862,113041,NaN,76594107.0,106.0,1995-12-08,1995,en,Comedy|Family,United States of America,Touchstone Pictures|Sandollar Productions,Charles Shyer,Steve Martin|Diane Keaton|Martin Short|Kimberl...,Nancy Meyers|Charles Shyer,Just when George Banks has recovered from his ...,daughter|baby|parent child relationship|midlif...
2,7,11860,114319,58000000.0,53672080.0,127.0,1995-12-15,1995,en,Romance|Drama|Comedy,Germany|United States of America,Paramount Pictures|Constellation Films|Mirage ...,Sydney Pollack,Harrison Ford|Julia Ormond|Greg Kinnear|Nancy ...,Barbara Benedek|David Rayfiel,"After her return from school in Paris, a playb...","chauffeur|sibling relationship|paris, france|t..."
3,6,949,113277,60000000.0,187400000.0,170.0,1995-12-15,1995,en,Crime|Drama|Action,United States of America,Warner Bros. Pictures|Regency Enterprises|Forw...,Michael Mann,Al Pacino|Robert De Niro|Val Kilmer|Jon Voight...,Michael Mann,Obsessive master thief Neil McCauley leads a t...,robbery|chase|obsession|detective|heist|remake...
4,1,862,114709,30000000.0,401157969.0,81.0,1995-11-22,1995,en,Family|Comedy|Animation|Adventure,United States of America,Pixar,John Lasseter,Tom Hanks|Tim Allen|Don Rickles|Jim Varney|Wal...,Alec Sokolow|Joel Cohen|Joss Whedon|Andrew Sta...,"Led by Woody, Andy's toys live happily in his ...",rescue|friendship|mission|jealousy|villain|bul...


In [13]:
# Join selected TMDB metadata back to MovieLens movies.
# movieId is the shared key across movies.csv, ratings.csv, and links.csv.
movies_enriched_df = movies_df.merge(
    tmdb_metadata_df,
    on="movieId",
    how="left"
)

movies_enriched_df.to_csv(ENRICHED_MOVIES_FILE, index=False)

print("Saved enriched movies file:", ENRICHED_MOVIES_FILE)
print("Rows:", len(movies_enriched_df))
movies_enriched_df.head()

Saved enriched movies file: /content/gdrive/MyDrive/Colab Notebooks/Data/DSC_511_PROJECT/enrichment_cache/movies_enriched_selected.csv
Rows: 86537


,movieId,title,genres,tmdb_id,imdb_id,budget,revenue,runtime,release_date,release_year,original_language,genres_tmdb,production_countries,production_companies,director,top_cast,writer,overview,keywords
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,862.0,114709.0,30000000.0,401157969.0,81.0,1995-11-22,1995.0,en,Family|Comedy|Animation|Adventure,United States of America,Pixar,John Lasseter,Tom Hanks|Tim Allen|Don Rickles|Jim Varney|Wal...,Alec Sokolow|Joel Cohen|Joss Whedon|Andrew Sta...,"Led by Woody, Andy's toys live happily in his ...",rescue|friendship|mission|jealousy|villain|bul...
1,2,Jumanji (1995),Adventure|Children|Fantasy,8844.0,113497.0,65000000.0,262821940.0,104.0,1995-12-15,1995.0,en,Adventure|Fantasy|Family,United States of America,TriStar Pictures|Interscope Communications|Tei...,Joe Johnston,Robin Williams|Kirsten Dunst|Bradley Pierce|Bo...,Jonathan Hensleigh|Greg Taylor|Jim Strain,When siblings Judy and Peter discover an encha...,giant insect|board game|disappearance|jungle|r...
2,3,Grumpier Old Men (1995),Comedy|Romance,15602.0,113228.0,25000000.0,71518503.0,101.0,1995-12-22,1995.0,en,Romance|Comedy,United States of America,Lancaster Gate|Warner Bros. Pictures,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret|Sophia ...,Mark Steven Johnson,A family wedding reignites the ancient feud be...,fishing|sequel|old man|best friend|wedding|ita...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,31357.0,114885.0,16000000.0,81452156.0,127.0,1995-12-22,1995.0,en,Comedy|Drama|Romance,United States of America,20th Century Fox,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine|...,Terry McMillan|Ronald Bass,"Cheated on, mistreated and stepped on, the wom...",based on novel or book|single mother|divorce|a...
4,5,Father of the Bride Part II (1995),Comedy,11862.0,113041.0,NaN,76594107.0,106.0,1995-12-08,1995.0,en,Comedy|Family,United States of America,Touchstone Pictures|Sandollar Productions,Charles Shyer,Steve Martin|Diane Keaton|Martin Short|Kimberl...,Nancy Meyers|Charles Shyer,Just when George Banks has recovered from his ...,daughter|baby|parent child relationship|midlif...


In [14]:
# Quick data quality summary for the selected metadata fields.
metadata_columns = [col for col in FIELDNAMES if col in movies_enriched_df.columns]
movies_enriched_df[metadata_columns].isna().sum().sort_values(ascending=False)

,0
budget,78252
revenue,77606
keywords,68914
writer,67829
production_companies,67634
production_countries,66989
top_cast,66882
runtime,66689
genres_tmdb,66688
director,66667
